# 11. Model Architecture Inspection

Phase test này không huấn luyện mô hình. Mục tiêu là kiểm tra cấu trúc các model chính đã dùng trong đồ án:

- `SimpleCNN`: baseline CNN tự xây dựng.
- `MobileNetV2`: pretrained backbone hiện đại dùng ở Phase 08/09.
- `EfficientNet-B0`: pretrained backbone hiện đại dùng để so sánh với MobileNetV2.

Notebook dùng `weights=None` cho MobileNetV2/EfficientNet-B0 để chỉ kiểm tra kiến trúc và không cần tải pretrained weights.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models

try:
    from IPython.display import display
except ImportError:
    display = print

def bootstrap_project_dir() -> Path:
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidates = [root, root / 'plan']
        for candidate in candidates:
            if (candidate / 'src').exists():
                if str(candidate) not in sys.path:
                    sys.path.append(str(candidate))
                from src.paths import find_project_dir
                return find_project_dir(candidate)
    raise FileNotFoundError('Không tìm thấy thư mục project plan chứa src/. Hãy chạy notebook từ repo hoặc thư mục plan.')


PROJECT_DIR = bootstrap_project_dir()

from src.data import ProjectPaths, ensure_directories, load_metadata
from src.models import get_simple_cnn

PATHS = ProjectPaths(PROJECT_DIR)
METADATA_DIR = PATHS.metadata_dir
RESULTS_DIR = PATHS.results_dir('11_model_architecture_inspection')
TABLE_DIR = RESULTS_DIR / 'tables'
REPORT_DIR = RESULTS_DIR / 'reports'

ensure_directories(RESULTS_DIR, TABLE_DIR, REPORT_DIR)

if (METADATA_DIR / 'class_map.csv').exists():
    _, _, _, class_map_df = load_metadata(METADATA_DIR)
    NUM_CLASSES = len(class_map_df)
else:
    NUM_CLASSES = 38

print(f'Số class dùng để dựng classifier head: {NUM_CLASSES}')


Số class dùng để dựng classifier head: 38


# 01. Dựng 3 model chính để inspect

Cell này dựng model theo đúng vai trò nghiên cứu nhưng không train. Với model torchvision, `weights=None` vẫn giữ nguyên kiến trúc, chỉ không nạp trọng số ImageNet.

In [ ]:
simple_cnn = get_simple_cnn(num_classes=NUM_CLASSES)

mobilenetv2 = models.mobilenet_v2(weights=None)
mn_in_features = mobilenetv2.classifier[1].in_features
mobilenetv2.classifier[1] = nn.Linear(mn_in_features, NUM_CLASSES)

efficientnet_b0 = models.efficientnet_b0(weights=None)
eff_in_features = efficientnet_b0.classifier[1].in_features
efficientnet_b0.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(eff_in_features, NUM_CLASSES),
)

models_to_inspect = {
    'simple_cnn': simple_cnn,
    'mobilenetv2': mobilenetv2,
    'efficientnet_b0': efficientnet_b0,
}

for model_name, model in models_to_inspect.items():
    print(f'\n===== {model_name} =====')
    print(model)



===== simple_cnn =====
SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(12, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(12, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(24, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(48, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_

# 02. Helper tóm tắt layer và parameter

Các helper này tạo bảng inspect thống nhất cho cả custom CNN và model torchvision.

In [ ]:
def readable_layer_type(module: nn.Module) -> str:
    class_name = module.__class__.__name__
    if class_name == 'Conv2dNormActivation':
        return 'ConvBNActivation'
    return class_name


def count_parameters(module: nn.Module, trainable_only: bool = False) -> int:
    parameters = module.parameters()
    if trainable_only:
        parameters = [p for p in parameters if p.requires_grad]
    return sum(p.numel() for p in parameters)


def first_child_types(module: nn.Module) -> str:
    child_names = [readable_layer_type(child) for child in module.children()]
    return ' -> '.join(child_names[:8])


def summarize_sequence(sequence: nn.Module, model_name: str, component: str) -> pd.DataFrame:
    records = []
    for index, layer in enumerate(sequence):
        layer_type = readable_layer_type(layer)
        records.append({
            'model': model_name,
            'component': component,
            'index': index,
            'layer_label': f'[{index}] {layer_type}',
            'layer_type': layer_type,
            'children': first_child_types(layer),
            'parameters': count_parameters(layer),
            'trainable_parameters': count_parameters(layer, trainable_only=True),
        })
    return pd.DataFrame(records)


def summarize_top_children(model: nn.Module, model_name: str) -> pd.DataFrame:
    records = []
    for name, child in model.named_children():
        records.append({
            'model': model_name,
            'component': name,
            'layer_type': readable_layer_type(child),
            'children': first_child_types(child),
            'parameters': count_parameters(child),
            'trainable_parameters': count_parameters(child, trainable_only=True),
        })
    return pd.DataFrame(records)


def summarize_model_parameters(models_dict: dict[str, nn.Module]) -> pd.DataFrame:
    records = []
    for model_name, model in models_dict.items():
        records.append({
            'model': model_name,
            'model_type': readable_layer_type(model),
            'total_parameters': count_parameters(model),
            'trainable_parameters': count_parameters(model, trainable_only=True),
        })
    return pd.DataFrame(records)


# 03. Inspect SimpleCNN baseline

SimpleCNN là baseline tự xây dựng, gồm `features` extractor nhỏ và `classifier` head. Model này dùng để kiểm tra baseline trước khi chuyển sang pretrained backbone.

In [ ]:
simple_feature_df = summarize_sequence(simple_cnn.features, 'simple_cnn', 'features')
simple_classifier_df = summarize_sequence(simple_cnn.classifier, 'simple_cnn', 'classifier')
simple_top_df = summarize_top_children(simple_cnn, 'simple_cnn')

simple_feature_df.to_csv(TABLE_DIR / 'simple_cnn_feature_layers.csv', index=False)
simple_classifier_df.to_csv(TABLE_DIR / 'simple_cnn_classifier_layers.csv', index=False)
simple_top_df.to_csv(TABLE_DIR / 'simple_cnn_top_components.csv', index=False)

print('SimpleCNN features:')
for layer_label in simple_feature_df['layer_label']:
    print(layer_label)

display(simple_feature_df)
display(simple_classifier_df)
display(simple_top_df)


SimpleCNN features:
[0] Conv2d
[1] BatchNorm2d
[2] ReLU
[3] MaxPool2d
[4] Conv2d
[5] BatchNorm2d
[6] ReLU
[7] MaxPool2d
[8] Conv2d
[9] BatchNorm2d
[10] ReLU
[11] MaxPool2d
[12] Conv2d
[13] BatchNorm2d
[14] ReLU
[15] AdaptiveAvgPool2d


,model,component,index,layer_label,layer_type,children,parameters,trainable_parameters
0,simple_cnn,features,0,[0] Conv2d,Conv2d,,336,336
1,simple_cnn,features,1,[1] BatchNorm2d,BatchNorm2d,,24,24
2,simple_cnn,features,2,[2] ReLU,ReLU,,0,0
3,simple_cnn,features,3,[3] MaxPool2d,MaxPool2d,,0,0
4,simple_cnn,features,4,[4] Conv2d,Conv2d,,2616,2616
5,simple_cnn,features,5,[5] BatchNorm2d,BatchNorm2d,,48,48
6,simple_cnn,features,6,[6] ReLU,ReLU,,0,0
7,simple_cnn,features,7,[7] MaxPool2d,MaxPool2d,,0,0
8,simple_cnn,features,8,[8] Conv2d,Conv2d,,10416,10416
9,simple_cnn,features,9,[9] BatchNorm2d,BatchNorm2d,,96,96


,model,component,index,layer_label,layer_type,children,parameters,trainable_parameters
0,simple_cnn,classifier,0,[0] Linear,Linear,,6208,6208
1,simple_cnn,classifier,1,[1] ReLU,ReLU,,0,0
2,simple_cnn,classifier,2,[2] Dropout,Dropout,,0,0
3,simple_cnn,classifier,3,[3] Linear,Linear,,2470,2470


,model,component,layer_type,children,parameters,trainable_parameters
0,simple_cnn,features,Sequential,Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d -> ...,55296,55296
1,simple_cnn,classifier,Sequential,Linear -> ReLU -> Dropout -> Linear,8678,8678


# 04. Inspect MobileNetV2 backbone

MobileNetV2 có `features` extractor gồm 19 top-level blocks. Trong torchvision mới, block đầu/cuối thường hiển thị là `Conv2dNormActivation`; trong báo cáo có thể gọi ngắn là `ConvBNActivation` hoặc `ConvBNReLU6`. Các block `[1]` đến `[17]` là `InvertedResidual`.

In [ ]:
mobilenet_feature_df = summarize_sequence(mobilenetv2.features, 'mobilenetv2', 'features')
mobilenet_classifier_df = summarize_sequence(mobilenetv2.classifier, 'mobilenetv2', 'classifier')
mobilenet_top_df = summarize_top_children(mobilenetv2, 'mobilenetv2')

mobilenet_feature_df.to_csv(TABLE_DIR / 'mobilenetv2_feature_layers.csv', index=False)
mobilenet_classifier_df.to_csv(TABLE_DIR / 'mobilenetv2_classifier_layers.csv', index=False)
mobilenet_top_df.to_csv(TABLE_DIR / 'mobilenetv2_top_components.csv', index=False)

print('MobileNetV2 features:')
for layer_label in mobilenet_feature_df['layer_label']:
    print(layer_label)

display(mobilenet_feature_df)
display(mobilenet_classifier_df)
display(mobilenet_top_df)


MobileNetV2 features:
[0] ConvBNActivation
[1] InvertedResidual
[2] InvertedResidual
[3] InvertedResidual
[4] InvertedResidual
[5] InvertedResidual
[6] InvertedResidual
[7] InvertedResidual
[8] InvertedResidual
[9] InvertedResidual
[10] InvertedResidual
[11] InvertedResidual
[12] InvertedResidual
[13] InvertedResidual
[14] InvertedResidual
[15] InvertedResidual
[16] InvertedResidual
[17] InvertedResidual
[18] ConvBNActivation


,model,component,index,layer_label,layer_type,children,parameters,trainable_parameters
0,mobilenetv2,features,0,[0] ConvBNActivation,ConvBNActivation,Conv2d -> BatchNorm2d -> ReLU6,928,928
1,mobilenetv2,features,1,[1] InvertedResidual,InvertedResidual,Sequential,896,896
2,mobilenetv2,features,2,[2] InvertedResidual,InvertedResidual,Sequential,5136,5136
3,mobilenetv2,features,3,[3] InvertedResidual,InvertedResidual,Sequential,8832,8832
4,mobilenetv2,features,4,[4] InvertedResidual,InvertedResidual,Sequential,10000,10000
5,mobilenetv2,features,5,[5] InvertedResidual,InvertedResidual,Sequential,14848,14848
6,mobilenetv2,features,6,[6] InvertedResidual,InvertedResidual,Sequential,14848,14848
7,mobilenetv2,features,7,[7] InvertedResidual,InvertedResidual,Sequential,21056,21056
8,mobilenetv2,features,8,[8] InvertedResidual,InvertedResidual,Sequential,54272,54272
9,mobilenetv2,features,9,[9] InvertedResidual,InvertedResidual,Sequential,54272,54272


,model,component,index,layer_label,layer_type,children,parameters,trainable_parameters
0,mobilenetv2,classifier,0,[0] Dropout,Dropout,,0,0
1,mobilenetv2,classifier,1,[1] Linear,Linear,,48678,48678


,model,component,layer_type,children,parameters,trainable_parameters
0,mobilenetv2,features,Sequential,ConvBNActivation -> InvertedResidual -> Invert...,2223872,2223872
1,mobilenetv2,classifier,Sequential,Dropout -> Linear,48678,48678


# 05. Inspect EfficientNet-B0 backbone

EfficientNet-B0 là model pretrained thứ hai trong Phase 08. Khác MobileNetV2, backbone của EfficientNet-B0 gồm nhiều `MBConv`/`FusedMBConv` block trong `features`, thể hiện compound scaling và squeeze-excitation trong các stage.

In [ ]:
efficientnet_feature_df = summarize_sequence(efficientnet_b0.features, 'efficientnet_b0', 'features')
efficientnet_classifier_df = summarize_sequence(efficientnet_b0.classifier, 'efficientnet_b0', 'classifier')
efficientnet_top_df = summarize_top_children(efficientnet_b0, 'efficientnet_b0')

efficientnet_feature_df.to_csv(TABLE_DIR / 'efficientnet_b0_feature_layers.csv', index=False)
efficientnet_classifier_df.to_csv(TABLE_DIR / 'efficientnet_b0_classifier_layers.csv', index=False)
efficientnet_top_df.to_csv(TABLE_DIR / 'efficientnet_b0_top_components.csv', index=False)

print('EfficientNet-B0 features:')
for layer_label in efficientnet_feature_df['layer_label']:
    print(layer_label)

display(efficientnet_feature_df)
display(efficientnet_classifier_df)
display(efficientnet_top_df)


EfficientNet-B0 features:
[0] ConvBNActivation
[1] Sequential
[2] Sequential
[3] Sequential
[4] Sequential
[5] Sequential
[6] Sequential
[7] Sequential
[8] ConvBNActivation


,model,component,index,layer_label,layer_type,children,parameters,trainable_parameters
0,efficientnet_b0,features,0,[0] ConvBNActivation,ConvBNActivation,Conv2d -> BatchNorm2d -> SiLU,928,928
1,efficientnet_b0,features,1,[1] Sequential,Sequential,MBConv,1448,1448
2,efficientnet_b0,features,2,[2] Sequential,Sequential,MBConv -> MBConv,16714,16714
3,efficientnet_b0,features,3,[3] Sequential,Sequential,MBConv -> MBConv,46640,46640
4,efficientnet_b0,features,4,[4] Sequential,Sequential,MBConv -> MBConv -> MBConv,242930,242930
5,efficientnet_b0,features,5,[5] Sequential,Sequential,MBConv -> MBConv -> MBConv,543148,543148
6,efficientnet_b0,features,6,[6] Sequential,Sequential,MBConv -> MBConv -> MBConv -> MBConv,2026348,2026348
7,efficientnet_b0,features,7,[7] Sequential,Sequential,MBConv,717232,717232
8,efficientnet_b0,features,8,[8] ConvBNActivation,ConvBNActivation,Conv2d -> BatchNorm2d -> SiLU,412160,412160


,model,component,index,layer_label,layer_type,children,parameters,trainable_parameters
0,efficientnet_b0,classifier,0,[0] Dropout,Dropout,,0,0
1,efficientnet_b0,classifier,1,[1] Linear,Linear,,48678,48678


,model,component,layer_type,children,parameters,trainable_parameters
0,efficientnet_b0,features,Sequential,ConvBNActivation -> Sequential -> Sequential -...,4007548,4007548
1,efficientnet_b0,avgpool,AdaptiveAvgPool2d,,0,0
2,efficientnet_b0,classifier,Sequential,Dropout -> Linear,48678,48678


# 06. Bảng so sánh tổng quan 3 model

Cell này tạo bảng parameter tổng quan để dùng trong phần phân tích trade-off giữa baseline nhỏ và pretrained backbone hiện đại.

In [ ]:
model_summary_df = summarize_model_parameters(models_to_inspect)
model_summary_df.to_csv(TABLE_DIR / 'three_model_parameter_summary.csv', index=False)
display(model_summary_df)

all_top_components_df = pd.concat(
    [simple_top_df, mobilenet_top_df, efficientnet_top_df],
    ignore_index=True,
)
all_top_components_df.to_csv(TABLE_DIR / 'three_model_top_components.csv', index=False)
display(all_top_components_df)


,model,model_type,total_parameters,trainable_parameters
0,simple_cnn,SimpleCNN,63974,63974
1,mobilenetv2,MobileNetV2,2272550,2272550
2,efficientnet_b0,EfficientNet,4056226,4056226


,model,component,layer_type,children,parameters,trainable_parameters
0,simple_cnn,features,Sequential,Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d -> ...,55296,55296
1,simple_cnn,classifier,Sequential,Linear -> ReLU -> Dropout -> Linear,8678,8678
2,mobilenetv2,features,Sequential,ConvBNActivation -> InvertedResidual -> Invert...,2223872,2223872
3,mobilenetv2,classifier,Sequential,Dropout -> Linear,48678,48678
4,efficientnet_b0,features,Sequential,ConvBNActivation -> Sequential -> Sequential -...,4007548,4007548
5,efficientnet_b0,avgpool,AdaptiveAvgPool2d,,0,0
6,efficientnet_b0,classifier,Sequential,Dropout -> Linear,48678,48678


# 07. Xuất ghi chú phân tích

Report ngắn này dùng được trực tiếp trong phần giải thích kiến trúc.

In [8]:
simple_lines = '\n'.join(simple_feature_df['layer_label'].tolist())
mobilenet_lines = '\n'.join(mobilenet_feature_df['layer_label'].tolist())
efficientnet_lines = '\n'.join(efficientnet_feature_df['layer_label'].tolist())
summary_markdown = model_summary_df.to_string(index=False)

report_text = f'''# Phase 11: Model Architecture Inspection

Phase này kiểm tra cấu trúc 3 nhóm model chính của đồ án: SimpleCNN, MobileNetV2 và EfficientNet-B0. Notebook không train model và dùng `weights=None` cho model torchvision để tránh phụ thuộc pretrained download.

## Parameter Summary

{summary_markdown}

## SimpleCNN features

{simple_lines}

SimpleCNN là baseline tự xây dựng, có feature extractor nhỏ gồm các tầng convolution, batch normalization, ReLU, pooling và classifier MLP. Vai trò của SimpleCNN là baseline kiểm soát trước khi chuyển sang pretrained backbone.

## MobileNetV2 features

{mobilenet_lines}

MobileNetV2 gồm 19 top-level blocks trong `features`. Các block `[1]` đến `[17]` là `InvertedResidual`, đây là phần cốt lõi của MobileNetV2. Block đầu và cuối là convolution + normalization + activation; trong torchvision mới tên là `Conv2dNormActivation`, có thể ghi ngắn trong báo cáo là `ConvBNActivation` hoặc `ConvBNReLU6`.

## EfficientNet-B0 features

{efficientnet_lines}

EfficientNet-B0 có feature extractor theo các stage EfficientNet, gồm stem convolution, các nhóm MBConv/FusedMBConv và final convolution block. Model này được dùng ở Phase 08 để so sánh với MobileNetV2 trong nhóm pretrained CNN hiện đại.

## Ý nghĩa phân tích

SimpleCNN nhỏ và dễ kiểm soát nhưng năng lực biểu diễn thấp hơn pretrained backbones. MobileNetV2 dùng inverted residual bottleneck nên nhẹ và phù hợp mobile/edge. EfficientNet-B0 dùng thiết kế EfficientNet với scaling cân bằng depth/width/resolution, thường mạnh hơn nhưng cấu trúc phức tạp hơn. Việc inspect 3 model giúp giải thích vì sao Phase 08 chuyển từ baseline tự xây dựng sang pretrained backbone hiện đại.
'''

(REPORT_DIR / 'model_architecture_inspection.md').write_text(report_text, encoding='utf-8')
(REPORT_DIR / 'model_architecture_inspection.txt').write_text(report_text, encoding='utf-8')

print(report_text)
print(f'Đã lưu bảng tại: {TABLE_DIR}')
print(f'Đã lưu report tại: {REPORT_DIR}')


# Phase 11: Model Architecture Inspection

Phase này kiểm tra cấu trúc 3 nhóm model chính của đồ án: SimpleCNN, MobileNetV2 và EfficientNet-B0. Notebook không train model và dùng `weights=None` cho model torchvision để tránh phụ thuộc pretrained download.

## Parameter Summary

          model   model_type  total_parameters  trainable_parameters
     simple_cnn    SimpleCNN             63974                 63974
    mobilenetv2  MobileNetV2           2272550               2272550
efficientnet_b0 EfficientNet           4056226               4056226

## SimpleCNN features

[0] Conv2d
[1] BatchNorm2d
[2] ReLU
[3] MaxPool2d
[4] Conv2d
[5] BatchNorm2d
[6] ReLU
[7] MaxPool2d
[8] Conv2d
[9] BatchNorm2d
[10] ReLU
[11] MaxPool2d
[12] Conv2d
[13] BatchNorm2d
[14] ReLU
[15] AdaptiveAvgPool2d

SimpleCNN là baseline tự xây dựng, có feature extractor nhỏ gồm các tầng convolution, batch normalization, ReLU, pooling và classifier MLP. Vai trò của SimpleCNN là baseline kiểm soát trước khi chuyển sang 